In [ ]:
%pip install scikit-learn==1.7.2 xgboost

In [ ]:
%pip install pandas numpy scikit-learn==1.7.2 xgboost

In [ ]:
import sklearn
print(sklearn.__version__)      

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error as rmse

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df = pd.read_csv(url).fillna(0)       # HW นี้ใช้ทุกคอลัมน์ ต่างจาก HW2 ที่ใช้แค่ 4

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train.pop('fuel_efficiency_mpg').values    # pop = ดึง y ออกมา และลบคอลัมน์ออกจาก df ในคำสั่งเดียว
y_val = df_val.pop('fuel_efficiency_mpg').values

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))
features = list(dv.get_feature_names_out())


Q1 Tree feature

In [ ]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)
print(export_text(dt, feature_names=features))

Q2 + Q3 Random Forest Multiple size (Choose RMSE n=10, Number of estimators that give smallest value)

In [ ]:
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    print(n, round(rmse(y_val, rf.predict(X_val)), 3))

Q4 Find the best max_depth (Choose Number of estimators that give smallest value)

In [ ]:
for depth in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        scores.append(rmse(y_val, rf.predict(X_val)))
    print(depth, round(np.mean(scores), 4))

Q5 Feature importance (high = importance)

In [ ]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
importances.head(10)

Q6 XGBoost compare with eta (Choose the smallest)

In [ ]:
import xgboost as xgb

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, 'train'), (dval, 'val')]

for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }
    model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                      evals=watchlist, verbose_eval=False)
    print(eta, round(rmse(y_val, model.predict(dval)), 4))